In [5]:
import os
import torch
import sys
sys.path.append("../../")  # go to parent dir
import transformers
transformers.logging.set_verbosity_error()

# Import libraries
import einops
from transformer_lens import HookedTransformer
from transformer_lens.utils import get_device

# Turn off automatic differentiation to save memory
torch.set_grad_enabled(False)
device = get_device()

# Load model
MODEL = "meta-llama/Llama-3.2-1B-Instruct"
#MODEL = "meta-llama/Llama-3.1-8B-Instruct"
model = HookedTransformer.from_pretrained(MODEL, device=device, torch_dtype=torch.bfloat16)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loaded pretrained model meta-llama/Llama-3.2-1B-Instruct into HookedTransformer


In [6]:
# Define IOI prompt (reconstructed to match the report's positional tokens:
# pos1=After, pos3=and, pos4=Matt, pos12=Brian, pos16=to; see ioi_report.txt)
prompt = "After Brian and Matt went to the store that day, Brian handed the drink to"

tokens = model.to_tokens(prompt, prepend_bos=True)          # [1, seq]
logits, cache = model.run_with_cache(tokens)
str_tokens = model.to_str_tokens(prompt, prepend_bos=True)

print(f"seq_len = {tokens.shape[1]} (report expects 17)")
for i, t in enumerate(str_tokens):
    print(f"{i:2d}: {t!r}")
# NOTE: report positions index THIS sequence. If tokenization differs from the
# report's, tweak the prompt so pos 1/3/4/12/16 land on After/and/Matt/Brian/to.

seq_len = 17 (report expects 17)
 0: '<|begin_of_text|>'
 1: 'After'
 2: ' Brian'
 3: ' and'
 4: ' Matt'
 5: ' went'
 6: ' to'
 7: ' the'
 8: ' store'
 9: ' that'
10: ' day'
11: ','
12: ' Brian'
13: ' handed'
14: ' the'
15: ' drink'
16: ' to'


## Logit lens + OV-circuit "translator": references & caveats

**Logit lens** (nostalgebraist, 2020): read out any residual-stream vector by applying the model's final norm and unembedding, `logits = W_U · ln_final(x)`. Here `model.unembed(model.ln_final(x))`.

**OV circuit** (Elhage et al. 2021, *A Mathematical Framework for Transformer Circuits*, transformer-circuits.pub/2021/framework): a head's contribution to the residual stream is `pattern · (x W_V) W_O`; the token-mixing (QK) and content-transform (OV) parts factor apart, with the per-head OV map `W_OV = W_V W_O` of shape `[d_model, d_model]`. The residual stream is additive — every head and MLP writes a vector into it.

**Translator caveats.** `M_k = I + Σ_h W_V[k,h] W_O[k,h]` approximates layer *k*'s linear action on the residual, and therefore:
- **ignores the attention pattern** (assumes each head reads its own position, i.e. pattern = identity), so it mis-counts real routing;
- **ignores MLP nonlinearities** entirely (MLP layers add no `M_k` term);
- **ignores RMSNorm rescaling** between layers.

It is a first-order, attention-pattern-free sketch of "where this vector is heading", not a faithful forward pass. Also note `resid_post[l]` already includes layer *l*'s own attention, so composing from `k=l` mildly double-counts layer *l* (start at `l+1` / use `resid_pre` for a stricter reading).

In [7]:
import re

# --- logit-lens helper: final norm + unembed, top-k tokens ---
def logit_lens(vec, k=10):
    x = vec.to(model.cfg.dtype).reshape(1, 1, -1)
    logits = model.unembed(model.ln_final(x))            # [1, 1, d_vocab]
    top = logits[0, 0].topk(k)
    return [(model.to_single_str_token(i.item()), round(v.item(), 2))
            for v, i in zip(top.values, top.indices)]

# Nodes of the positional tree at the end of ioi_report.txt, hardcoded as
# (name, token_position). Columns pos 1/3/4/12/16 = After/and/Matt/Brian/to.
NODES = [
    ('A0H12', 4), ('A0H13', 4), ('A0H21', 4), ('A0H23', 4), ('A0H24', 4),
    ('A0H27', 4), ('A0H6', 4), ('A0H7', 4), ('A0H9', 4), ('A1H16', 4),
    ('A0H11', 12), ('A0H19', 12), ('A0H24', 12), ('A0H26', 12), ('MLP0', 12),
    ('A1H22', 12), ('MLP1', 12), ('MLP2', 12), ('MLP3', 12), ('A4H16', 12),
    ('MLP4', 12), ('A5H8', 12), ('A5H9', 12), ('MLP5', 12), ('MLP6', 12),
    ('MLP7', 12), ('A8H17', 12), ('A8H19', 12), ('MLP8', 12), ('A9H18', 16),
    ('A9H27', 16), ('MLP9', 16), ('MLP10', 16), ('A12H15', 16), ('A12H2', 16),
    ('A14H12', 16), ('A14H14', 16), ('A14H28', 16), ('A14H30', 16), ('MLP14', 16),
    ('A15H7', 16), ('MLP15', 16),
]

# Decode each component from its residual write-vector AT ITS OWN token position.
for name, pos in NODES:
    tok = str_tokens[pos]
    m = re.fullmatch(r'A(\d+)H(\d+)', name)
    if m:
        L, H = int(m[1]), int(m[2])
        vec = cache["z", L][0, pos, H, :] @ model.W_O[L, H]   # head output -> resid, [d_model]
    else:
        L = int(name[3:])
        vec = cache["mlp_out", L][0, pos, :]                  # [d_model]
    print(f"{name:>7}@{pos:<3}({tok!r:>9}) -> {logit_lens(vec)}")

  A0H12@4  (  ' Matt') -> [(' of', 1.73), ('-of', 1.16), (' Of', 1.13), ('Of', 1.03), ('of', 0.95), ('_of', 0.89), ('(of', 0.85), (' của', 0.82), (' ant', 0.81), (' flow', 0.81)]
  A0H13@4  (  ' Matt') -> [('üst', 0.85), (' fut', 0.81), ('minster', 0.8), ('ós', 0.8), (' Truck', 0.79), (' necessity', 0.77), (' إلا', 0.77), (' Heroes', 0.77), ('anda', 0.76), ('acha', 0.76)]
  A0H21@4  (  ' Matt') -> [('-------------</', 11.19), ('----------</', 11.19), ('rubu', 10.62), ('-REAL', 10.5), ('\xa0PS', 10.25), ('tainment', 10.25), ('filme', 10.12), ('：</', 10.0), ('LOCKS', 9.88), (' ”\n\n', 9.88)]
  A0H23@4  (  ' Matt') -> [('alat', 4.56), (' yourselves', 4.25), (' brun', 4.22), ('lín', 4.22), (' ilan', 4.19), ('子は', 4.16), ('іти', 4.16), ('istingu', 4.12), ('tainment', 4.12), ('ONUS', 4.03)]
  A0H24@4  (  ' Matt') -> [(' States', 4.88), ('States', 4.44), ('uf', 4.44), (' states', 4.16), ('ayed', 3.81), ('reg', 3.72), ('apon', 3.72), ('teg', 3.66), ('ンテ', 3.61), (' dock', 3.59)]
  A0H27@4  (  

In [9]:
# --- OV-circuit translator: compose per-layer head-OV maps forward to the logits ---
I = torch.eye(model.cfg.d_model, dtype=torch.float32, device=model.W_U.device)

def ov_matrix(k):
    # M_k = I + sum_h W_V[k,h] @ W_O[k,h]  (attention-pattern-free OV action of layer k)
    wv = model.W_V[k].float()                        # [n_heads, d_model, d_head]
    wo = model.W_O[k].float()                        # [n_heads, d_head, d_model]
    return I + torch.einsum('hmd,hdn->mn', wv, wo)   # [d_model, d_model]

def translate_and_decode(resid_vec, l, k=10):
    vec = resid_vec.float()                          # row vector [d_model]
    for kk in range(l, model.cfg.n_layers):          # resid @ M_l @ ... @ M_15
        vec = vec @ ov_matrix(kk)
    return logit_lens(vec, k)

# --- demo: plain logit lens vs translator on the last-position residual, a few layers ---
pos = cache["resid_post", 0].shape[1] - 1            # final (prediction) position
for l in (6, 9, 12, 14, 15):
    resid = cache["resid_post", l][0, pos]           # [d_model]
    print(f"\n=== resid_post L{l} @ pos {pos} ===")
    print("  logit lens :", logit_lens(resid))
    print("  translator :", translate_and_decode(resid, l))  # caveat: k=l double-counts L{l}


=== resid_post L6 @ pos 16 ===
  logit lens : [(' carrying', 9.88), (' hand', 9.56), ('itional', 9.5), ('_hand', 9.44), ('—to', 9.0), ('主任', 8.62), ('ningen', 8.56), (' Hand', 8.5), (' assisting', 8.5), (' unto', 8.31)]
  translator : [('itional', 10.12), (' entreg', 9.38), ('—to', 9.19), (' unto', 9.06), (' Receiver', 9.0), (' transpose', 9.0), (' receiver', 9.0), (' hand', 8.94), (' Operation', 8.88), ('Transpose', 8.88)]

=== resid_post L9 @ pos 16 ===
  logit lens : [('bart', 9.94), ('なく', 8.88), (' Reception', 8.81), ('apo', 8.12), ('odont', 7.94), ('wipe', 7.91), ('держ', 7.88), (' handed', 7.81), ('illary', 7.78), ('lj', 7.72)]
  translator : [(' handed', 11.0), (' him', 10.88), (' handing', 9.31), (' unto', 8.56), ('-sm', 8.5), ('なく', 8.38), (' straight', 8.38), (' strangers', 8.06), (' waited', 7.97), (' automatically', 7.88)]

=== resid_post L12 @ pos 16 ===
  logit lens : [(' himself', 11.44), (' him', 10.44), (' somebody', 8.88), ('him', 8.75), (' teammate', 8.56), ('someo

In [10]:
# --- per-head: plain logit lens vs OV-translator, on each head's own write-vector ---
# The head writes into resid at layer L; the translator pushes that contribution
# forward through the remaining layers L+1..15 (no double-count of L's own attn).
for name, pos in NODES:
    m = re.fullmatch(r'A(\d+)H(\d+)', name)
    if not m:
        continue                                             # attention heads only
    L, H = int(m[1]), int(m[2])
    vec = cache["z", L][0, pos, H, :] @ model.W_O[L, H]      # head output -> resid, [d_model]
    plain = logit_lens(vec, k=6)
    trans = translate_and_decode(vec, L + 1, k=6)            # compose M_{L+1}..M_15
    print(f"{name:>7}@{pos:<3}({str_tokens[pos]!r:>9})")
    print(f"    plain : {', '.join(t for t, _ in plain)}")
    print(f"    transl: {', '.join(t for t, _ in trans)}")

  A0H12@4  (  ' Matt')
    plain :  of, -of,  Of, Of, of, _of
    transl:  of,  celebration,  celebr,  ce,  Of, -of
  A0H13@4  (  ' Matt')
    plain : üst,  fut, minster, ós,  Truck,  necessity
    transl:  Truck, nip, weed, üst, ustin,  resid
  A0H21@4  (  ' Matt')
    plain : ----------</, -------------</, rubu, -REAL,  PS, tainment
    transl:  --

,  PS, -REAL,  ”

, .':, _',
  A0H23@4  (  ' Matt')
    plain : alat,  yourselves, lín,  brun,  ilan, іти
    transl:  brun, tat, опол,  herself, jen,  yourselves
  A0H24@4  (  ' Matt')
    plain :  States, States, uf,  states, ayed, reg
    transl: teg, combe, States, 照,  naš, hone
  A0H27@4  (  ' Matt')
    plain : brids, inize, ,www, ografie, umblr, SetActive
    transl: -performance, asers,  Platforms, ,...

, anje, .uk
   A0H6@4  (  ' Matt')
    plain : ing, er, e, es, ic, ers
    transl: able, er, ing, ic, us, ors
   A0H7@4  (  ' Matt')
    plain :  dis, .baomidou, QUERY, шев, няти, etypes
    transl:  dis, kills, .baomidou, urtles,